downloading ucimlrepo<br>
since we will be using a health disease dataset from https://archive.ics.uci.edu/dataset/14/breast+cancer

In [ ]:
!pip install ucimlrepo

importing the modules.

In [ ]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

## importing the dataset from the site.

In [ ]:
# fetch dataset
breast_cancer = fetch_ucirepo(id=14)

# data (as pandas dataframes)
X = breast_cancer.data.features
y = breast_cancer.data.targets
df = pd.concat([X, y], axis=1)

# metadata
print(breast_cancer.metadata)
# variable information
print(breast_cancer.variables)

{'uci_id': 14, 'name': 'Breast Cancer', 'repository_url': 'https://archive.ics.uci.edu/dataset/14/breast+cancer', 'data_url': 'https://archive.ics.uci.edu/static/public/14/data.csv', 'abstract': 'This breast cancer domain was obtained from the University Medical Centre, Institute of Oncology, Ljubljana, Yugoslavia. This is one of three domains provided by the Oncology Institute that has repeatedly appeared in the machine learning literature. (See also lymphography and primary-tumor.)', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 286, 'num_features': 9, 'feature_types': ['Categorical'], 'demographics': ['Age'], 'target_col': ['Class'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1988, 'last_updated': 'Thu Mar 07 2024', 'dataset_doi': '10.24432/C51P4M', 'creators': ['Matjaz Zwitter', 'Milan Soklic'], 'intro_paper': None, 'additional_info': {'summary': 'Thi

In [ ]:
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 286 entries, 0 to 285
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          286 non-null    object
 1   menopause    286 non-null    object
 2   tumor-size   286 non-null    object
 3   inv-nodes    286 non-null    object
 4   node-caps    278 non-null    object
 5   deg-malig    286 non-null    int64 
 6   breast       286 non-null    object
 7   breast-quad  285 non-null    object
 8   irradiat     286 non-null    object
 9   Class        286 non-null    object
dtypes: int64(1), object(9)
memory usage: 22.5+ KB
None
        deg-malig
count  286.000000
mean     2.048951
std      0.738217
min      1.000000
25%      2.000000
50%      2.000000
75%      3.000000
max      3.000000


## we will now clean and process the data.

In [ ]:
#checking if any column has mission values
print(df.isnull().sum())

age            0
menopause      0
tumor-size     0
inv-nodes      0
node-caps      8
deg-malig      0
breast         0
breast-quad    1
irradiat       0
Class          0
dtype: int64


#### `node-caps` and `breast-quad` have missing values (stored as '?') <br>
we will replace '?' with NaN, then fill both (categorical) with mode values.

In [ ]:
df['node-caps'] = df['node-caps'].replace('?', np.nan)
df['breast-quad'] = df['breast-quad'].replace('?', np.nan)

df['node-caps'] = df['node-caps'].fillna(df['node-caps'].mode()[0]) #mode for node-caps
print(f"null values in node-caps is replaced with {df['node-caps'].mode()[0]}")
df['breast-quad'] = df['breast-quad'].fillna(df['breast-quad'].mode()[0]) #mode for breast-quad
print(f"null values in breast-quad is replaced with {df['breast-quad'].mode()[0]}")

null values in node-caps is replaced with no
null values in breast-quad is replaced with left_low


In [ ]:
print("null values in:")
print(f"node-caps: {df['node-caps'].isnull().sum()}")
print(f"breast-quad: {df['breast-quad'].isnull().sum()}")

null values in:
node-caps: 0
breast-quad: 0


## Aggregation.

In [ ]:
aggregation = df.groupby('Class').agg({
    'deg-malig': 'mean'
})

print(aggregation)

                      deg-malig
Class                          
no-recurrence-events   1.905473
recurrence-events      2.388235


In [ ]:
aggregation = df.groupby('menopause').agg({
    'deg-malig': ['mean', 'min', 'max']
})

print(aggregation)

          deg-malig        
               mean min max
menopause                  
ge40       2.093023   1   3
lt40       1.714286   1   3
premeno    2.026667   1   3


#Discretization

In [ ]:
malig_bins = [0, 1, 2, 3]

malig_labels = [
    'Low',
    'Medium',
    'High'
]

df['malignancy_category'] = pd.cut(
    df['deg-malig'],
    bins=malig_bins,
    labels=malig_labels
)

print(df[['deg-malig', 'malignancy_category']].head())

   deg-malig malignancy_category
0          3                High
1          2              Medium
2          2              Medium
3          2              Medium
4          2              Medium


#Binarization

In [ ]:
df['node_caps_flag'] = (df['node-caps'] == 'yes').astype(int)
df['irradiat_flag'] = (df['irradiat'] == 'yes').astype(int)
df['recurrence'] = (df['Class'] == 'recurrence-events').astype(int)
print(df[['node-caps', 'node_caps_flag', 'irradiat', 'irradiat_flag', 'Class', 'recurrence']].head())

  node-caps  node_caps_flag irradiat  irradiat_flag                 Class  \
0        no               0       no              0  no-recurrence-events   
1        no               0       no              0  no-recurrence-events   
2        no               0       no              0  no-recurrence-events   
3        no               0       no              0  no-recurrence-events   
4        no               0       no              0  no-recurrence-events   

   recurrence  
0           0  
1           0  
2           0  
3           0  
4           0  


#Sampling
selecting a subset of the entire dataset.

In [ ]:
sample = df.sample(
    n=100,
    random_state=42
)

print(sample)

       age menopause tumor-size inv-nodes node-caps  deg-malig breast  \
9    40-49   premeno      20-24       0-2        no          2  right   
267  60-69      ge40      20-24     24-26       yes          3   left   
143  40-49   premeno      45-49       0-2        no          2   left   
212  40-49   premeno      30-34       0-2        no          3  right   
227  50-59   premeno      30-34       0-2        no          3  right   
..     ...       ...        ...       ...       ...        ...    ...   
57   50-59      ge40      9-May       0-2        no          2  right   
108  40-49   premeno      30-34       0-2        no          3  right   
272  40-49   premeno      15-19       0-2       yes          3  right   
206  50-59      ge40      30-34       0-2        no          3   left   
148  30-39   premeno      20-24       0-2        no          3   left   

    breast-quad irradiat                 Class malignancy_category  \
9       left_up       no  no-recurrence-events       